In [1]:
# Import libraries and other configurations
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 121

In [2]:
# Load the cleaned dataset
df = pd.read_csv(
    "../data/processed/cleaned_donor_data.csv",
    dtype={"donor_postal_code": "str"}
)

df.columns.tolist()

['donor_unique_id',
 'donor_postal_code',
 'donor_age',
 'gender_identity',
 'is_member_flag',
 'is_alumnus_flag',
 'is_parent_flag',
 'has_involvement_flag',
 'preferred_address_type',
 'has_email_flag',
 'consecutive_donor_years',
 'last_fiscal_year_donation',
 'donation_2_fiscal_years_ago',
 'donation_3_fiscal_years_ago',
 'donation_4_fiscal_years_ago',
 'donation_5_fiscal_years_ago',
 'current_fiscal_year_donation',
 'cumulative_donation_amount',
 'donor_indicator_flag']

In [3]:
# Verify dataset loaded correctly
print(f"Dataset Shape: {df.shape}")
df.sample(10, random_state=RANDOM_STATE)

Dataset Shape: (34403, 19)


,donor_unique_id,donor_postal_code,donor_age,gender_identity,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,preferred_address_type,has_email_flag,consecutive_donor_years,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount,donor_indicator_flag
33267,33371,42301.0,46,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
26595,26683,45620.0,42,Female,0,0,0,0,Home,0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
16653,16704,33427.0,28,Female,0,0,0,0,Home,0,0,0.00,120.00,0.00,0.00,0.00,0.00,120.00,1
21428,21498,54131.0,33,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1
16873,16924,90265.0,33,Male,0,1,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
19919,19982,54459.0,32,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
15478,15527,64686.0,42,Male,0,0,0,0,Home,0,8,0.00,0.00,10.00,0.00,0.00,0.00,"9,680.00",1
591,596,12067.0,42,Female,0,0,1,0,Home,0,0,0.00,100.00,1.00,0.00,0.00,0.00,101.00,1
1474,1483,90265.0,42,Female,0,1,0,0,Home,1,0,0.00,0.00,0.00,0.00,0.00,0.00,479.00,1
22846,22922,45856.0,36,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [4]:
# Verify dataset data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   donor_unique_id               34403 non-null  int64  
 1   donor_postal_code             34312 non-null  str    
 2   donor_age                     34403 non-null  int64  
 3   gender_identity               33912 non-null  str    
 4   is_member_flag                34403 non-null  int64  
 5   is_alumnus_flag               34403 non-null  int64  
 6   is_parent_flag                34403 non-null  int64  
 7   has_involvement_flag          34403 non-null  int64  
 8   preferred_address_type        30370 non-null  str    
 9   has_email_flag                34403 non-null  int64  
 10  consecutive_donor_years       34403 non-null  int64  
 11  last_fiscal_year_donation     34403 non-null  float64
 12  donation_2_fiscal_years_ago   34403 non-null  float64
 13  donation_3_f

In [5]:
# Data types summary
dtype_summary = (
    df.dtypes
      .astype(str)
      .value_counts()
      .reset_index()
      .style.hide(axis="index")
)

dtype_summary.columns = ["Data Type", "Count"]
dtype_summary

index,count
int64,9
float64,7
str,3


In [6]:
# Verify dataset missing values
missing_summary = (
    df.isna()
      .sum()
      .to_frame("Missing Count")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / len(df) * 100
)

missing_summary = (
    missing_summary
      .sort_values("Missing Count", ascending=False)
)

missing_summary

,Missing Count,Missing %
preferred_address_type,4033,11.72
gender_identity,491,1.43
donor_postal_code,91,0.26
donor_unique_id,0,0.00
last_fiscal_year_donation,0,0.00
cumulative_donation_amount,0,0.00
current_fiscal_year_donation,0,0.00
donation_5_fiscal_years_ago,0,0.00
donation_4_fiscal_years_ago,0,0.00
donation_3_fiscal_years_ago,0,0.00


## Dataset Overview
The finalized cleaned analytical dataset was successfully loaded and verified for feature engineering. The dataset contains 34,403 donor records and 19 features, consistent with the dataset used throughout Phase 2. Data types and missing values match expectations from the data cleaning process, and all required libraries were successfully imported. The only change made was explicitly defining `donor_postal_code` as a string. This was made because it was interpreting postal codes as a float and not a string.

In [7]:
# Define target-related column names
TARGET_SOURCE_COLUMN = "current_fiscal_year_donation"
TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

# Validate the target source before creating the binary target
assert pd.api.types.is_numeric_dtype(df[TARGET_SOURCE_COLUMN]), (
    f"{TARGET_SOURCE_COLUMN} must contain numeric values."
)

# Check for missing values
missing_target_values = df[TARGET_SOURCE_COLUMN].isna().sum()
print(f"Missing target-source values: {missing_target_values:,}")

assert missing_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{missing_target_values:,} missing values."
)

# Check for negative numbers
negative_target_values = (df[TARGET_SOURCE_COLUMN] < 0).sum()
print(f"Negative target-source values: {negative_target_values:,}")

assert negative_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{negative_target_values:,} negative values."
)

Missing target-source values: 0
Negative target-source values: 0


In [8]:
# Create the binary prediction target
df[TARGET_COLUMN] = (
    df[TARGET_SOURCE_COLUMN] > 0
).astype("int8")

print(f"Target column created: {TARGET_COLUMN}")

Target column created: target_current_fiscal_year_donor_flag


In [9]:
# Summarize the target distribution
target_summary = (
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

target_summary["target_label"] = target_summary[TARGET_COLUMN].map(
    {
        0: "Did not donate",
        1: "Donated",
    }
)

target_summary = target_summary[
    [
        TARGET_COLUMN,
        "target_label",
        "count",
        "percentage",
    ]
]

display(target_summary.style.hide(axis="index").format({"percentage": "{:.2f}%", "count": "{:,}"}))

target_current_fiscal_year_donor_flag,target_label,count,percentage
0,Did not donate,"32,499",94.47%
1,Donated,"1,904",5.53%


In [10]:
# Confirm that the target contains only binary values and matches its source column
expected_target_values = {0, 1}
actual_target_values = set(df[TARGET_COLUMN].unique())

assert actual_target_values == expected_target_values, (
    f"Unexpected target values found: {actual_target_values}"
)

# Confirm that the target matches its source column
target_mismatch_count = (
    df[TARGET_COLUMN]
    != (df[TARGET_SOURCE_COLUMN] > 0).astype("int8")
).sum()

assert target_mismatch_count == 0, (
    f"Found {target_mismatch_count:,} target-definition mismatches."
)

print("Target validation passed.")

Target validation passed.


## Prediction Target and Time Window

The classification target is `target_current_fiscal_year_donor_flag`, which identifies whether an individual donated during the current fiscal year.

The target is derived from `current_fiscal_year_donation`:

- `0`: The individual donated $0 during the current fiscal year
  
- `1`: The individual donated more than $0 during the current fiscal year

The target distribution shows that 1,904 individuals, or 5.53% of the dataset, donated during the current fiscal year. The remaining 32,499 individuals, or 94.47%, did not donate. This indicates a substantial class imbalance that will need to be considered during model evaluation and training.

The intended prediction setup uses the five completed fiscal years before the current fiscal year as the historical observation window. The current fiscal year serves as the outcome period.

The model will therefore use information available before the beginning of the current fiscal year to predict whether an individual donates during that fiscal year.

The original `current_fiscal_year_donation` column will remain in the working dataset temporarily for validation, but it must not be included in the predictor matrix because it directly determines the target.

Until additional documentation is available:

* `cumulative_donation_amount` will not be used directly because it may include current-fiscal-year donations
* `donor_indicator_flag` will not be used as a predictor until its calculation timing and logic are confirmed
* `consecutive_donor_years` will not be used until it is confirmed that it excludes current-fiscal-year activity
* Demographic and engagement fields will be treated as provisionally available
* The exact fiscal-year dates and dataset extraction date should be confirmed before the final model is deployed

The timing and leakage status of all source variables will be evaluated during the formal leakage audit.


In [11]:
# Define leakage audit classifications
VALID_LEAKAGE_CLASSIFICATIONS = {
    "Safe predictor",
    "Potential leakage",
    "Identifier",
    "Target",
    "Excluded feature",
}

# Document the leakage decision for every column
leakage_audit = pd.DataFrame(
    [
        {
            "column": "donor_unique_id",
            "classification": "Identifier",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "Unique record identifier with no meaningful predictive value."
            ),
            "required_action": (
                "Retain only for record tracking and exported predictions."
            ),
        },
        {
            "column": "donor_postal_code",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Geographic information should be available before the "
                "prediction period and does not directly reveal the target."
            ),
            "required_action": (
                "Evaluate cardinality, privacy, and fairness before modeling."
            ),
        },
        {
            "column": "donor_age",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Demographic information does not directly reveal whether "
                "the individual donated during the target period."
            ),
            "required_action": (
                "Document that the exact age reference date is unavailable."
            ),
        },
        {
            "column": "gender_identity",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Demographic information does not directly reveal the "
                "current-fiscal-year donation outcome."
            ),
            "required_action": (
                "Consolidate unknown and missing categories before modeling."
            ),
        },
        {
            "column": "is_member_flag",
            "classification": "Excluded feature",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "The field is constant in the cleaned dataset and therefore "
                "provides no predictive variation."
            ),
            "required_action": "Exclude from the modeling feature set.",
        },
        {
            "column": "is_alumnus_flag",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Alumni status is a relatively stable relationship attribute "
                "and does not directly reveal the target."
            ),
            "required_action": (
                "Retain unless later documentation identifies a timing issue."
            ),
        },
        {
            "column": "is_parent_flag",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Parent status is a relatively stable relationship attribute "
                "and does not directly reveal the target."
            ),
            "required_action": (
                "Retain unless later documentation identifies a timing issue."
            ),
        },
        {
            "column": "has_involvement_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The snapshot date is unknown, so the field may include "
                "involvement established during the target period."
            ),
            "required_action": (
                "Confirm that the value was available at the prediction cutoff."
            ),
        },
        {
            "column": "preferred_address_type",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Contact-preference information does not directly reveal "
                "current-fiscal-year donation behavior."
            ),
            "required_action": (
                "Standardize categories and handle missing values before modeling."
            ),
        },
        {
            "column": "has_email_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The field may reflect contact information added or updated "
                "during the target period."
            ),
            "required_action": (
                "Confirm that the value was available at the prediction cutoff."
            ),
        },
        {
            "column": "consecutive_donor_years",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The count may include donation activity from the current "
                "fiscal year."
            ),
            "required_action": (
                "Confirm the calculation period or reconstruct it from "
                "historical donation columns."
            ),
        },
        {
            "column": "last_fiscal_year_donation",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from the completed fiscal year "
                "before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_2_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_3_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_4_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_5_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "current_fiscal_year_donation",
            "classification": "Excluded feature",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "Directly determines the binary modeling target and would "
                "cause target leakage if used as a predictor."
            ),
            "required_action": (
                "Retain only for target construction and validation."
            ),
        },
        {
            "column": "cumulative_donation_amount",
            "classification": "Excluded feature",
            "model_treatment": "Exclude or reconstruct",
            "reason": (
                "The cumulative value may include the current-fiscal-year "
                "donation and therefore reveal target-period information."
            ),
            "required_action": (
                "Replace with a historical value calculated only from "
                "pre-cutoff donation data."
            ),
        },
        {
            "column": "donor_indicator_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The calculation logic and timing are unknown and may include "
                "current-fiscal-year donation behavior."
            ),
            "required_action": (
                "Confirm the source logic and reference date before use."
            ),
        },
        {
            "column": "target_current_fiscal_year_donor_flag",
            "classification": "Target",
            "model_treatment": "Use as modeling target",
            "reason": (
                "Binary outcome created from whether "
                "current_fiscal_year_donation is greater than zero."
            ),
            "required_action": (
                "Use as the target variable and never include it among predictors."
            ),
        },
    ]
)

display(leakage_audit.style.hide(axis="index"))

column,classification,model_treatment,reason,required_action
donor_unique_id,Identifier,Exclude from predictors,Unique record identifier with no meaningful predictive value.,Retain only for record tracking and exported predictions.
donor_postal_code,Safe predictor,Candidate predictor,Geographic information should be available before the prediction period and does not directly reveal the target.,"Evaluate cardinality, privacy, and fairness before modeling."
donor_age,Safe predictor,Candidate predictor,Demographic information does not directly reveal whether the individual donated during the target period.,Document that the exact age reference date is unavailable.
gender_identity,Safe predictor,Candidate predictor,Demographic information does not directly reveal the current-fiscal-year donation outcome.,Consolidate unknown and missing categories before modeling.
is_member_flag,Excluded feature,Exclude from predictors,The field is constant in the cleaned dataset and therefore provides no predictive variation.,Exclude from the modeling feature set.
is_alumnus_flag,Safe predictor,Candidate predictor,Alumni status is a relatively stable relationship attribute and does not directly reveal the target.,Retain unless later documentation identifies a timing issue.
is_parent_flag,Safe predictor,Candidate predictor,Parent status is a relatively stable relationship attribute and does not directly reveal the target.,Retain unless later documentation identifies a timing issue.
has_involvement_flag,Potential leakage,Exclude pending verification,"The snapshot date is unknown, so the field may include involvement established during the target period.",Confirm that the value was available at the prediction cutoff.
preferred_address_type,Safe predictor,Candidate predictor,Contact-preference information does not directly reveal current-fiscal-year donation behavior.,Standardize categories and handle missing values before modeling.
has_email_flag,Potential leakage,Exclude pending verification,The field may reflect contact information added or updated during the target period.,Confirm that the value was available at the prediction cutoff.


In [12]:
# Confirm that each column appears exactly once
duplicate_audit_columns = leakage_audit["column"].duplicated().sum()

dataset_columns = set(df.columns)
audited_columns = set(leakage_audit["column"])

missing_audit_columns = sorted(dataset_columns - audited_columns)
unexpected_audit_columns = sorted(audited_columns - dataset_columns)

invalid_classifications = sorted(
    set(leakage_audit["classification"])
    - VALID_LEAKAGE_CLASSIFICATIONS
)

assert duplicate_audit_columns == 0, (
    f"The leakage audit contains "
    f"{duplicate_audit_columns} duplicate column entries."
)

assert not missing_audit_columns, (
    f"Columns missing from the leakage audit: "
    f"{missing_audit_columns}"
)

assert not unexpected_audit_columns, (
    f"Audit columns not found in the dataset: "
    f"{unexpected_audit_columns}"
)

assert not invalid_classifications, (
    f"Invalid classifications found: "
    f"{invalid_classifications}"
)

print("Leakage audit validation passed.")
print(f"Dataset columns audited: {len(leakage_audit)}")

Leakage audit validation passed.
Dataset columns audited: 20


In [13]:
# Summarize the num of columns in each classification
leakage_audit_summary = (
    leakage_audit["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="column_count")
)

display(leakage_audit_summary.style.hide(axis="index"))

classification,column_count
Safe predictor,11
Potential leakage,4
Excluded feature,3
Identifier,1
Target,1


In [14]:
# Reusable column lists for feature engineering and modeling
SAFE_PREDICTOR_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Safe predictor",
    "column",
].tolist()

POTENTIAL_LEAKAGE_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Potential leakage",
    "column",
].tolist()

IDENTIFIER_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Identifier",
    "column",
].tolist()

EXCLUDED_FEATURE_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Excluded feature",
    "column",
].tolist()

TARGET_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Target",
    "column",
].tolist()

print("Safe predictors:")
print(SAFE_PREDICTOR_COLUMNS)

print("\nPotential leakage columns:")
print(POTENTIAL_LEAKAGE_COLUMNS)

print("\nIdentifier columns:")
print(IDENTIFIER_COLUMNS)

print("\nExcluded features:")
print(EXCLUDED_FEATURE_COLUMNS)

print("\nTarget-related columns:")
print(TARGET_COLUMNS)

Safe predictors:
['donor_postal_code', 'donor_age', 'gender_identity', 'is_alumnus_flag', 'is_parent_flag', 'preferred_address_type', 'last_fiscal_year_donation', 'donation_2_fiscal_years_ago', 'donation_3_fiscal_years_ago', 'donation_4_fiscal_years_ago', 'donation_5_fiscal_years_ago']

Potential leakage columns:
['has_involvement_flag', 'has_email_flag', 'consecutive_donor_years', 'donor_indicator_flag']

Identifier columns:
['donor_unique_id']

Excluded features:
['is_member_flag', 'current_fiscal_year_donation', 'cumulative_donation_amount']

Target-related columns:
['target_current_fiscal_year_donor_flag']


## Feature Leakage Assessment

A formal leakage audit was completed for all 20 columns currently in the working dataset, including the newly created target variable.

Each column was assigned one of five classifications:

- Safe predictor: Available before the prediction period and does not directly reveal the target
- Potential leakage: May be usable, but its calculation timing or snapshot date must be verified
- Identifier: Used only for record tracking and not as a predictor
- Target: The binary outcome the model will predict
- Excluded feature: Must not be included in the predictor set because it is uninformative or directly risks leakage

The audit identified 11 safe predictors, consisting primarily of demographic, contact-preference, geographic, and completed historical donation variables.

The following four columns were classified as potential leakage:

- `has_involvement_flag`
- `has_email_flag`
- `consecutive_donor_years`
- `donor_indicator_flag`

`has_involvement_flag` and `has_email_flag` may be useful predictors, but their snapshot dates are unknown. They should remain excluded from the initial modeling feature set until it is confirmed that their values were available before the prediction cutoff.

`consecutive_donor_years` may include donation behavior from the current fiscal year. It must either be verified as a historical-only measure or replaced with a streak feature reconstructed from the five completed historical donation years.

`donor_indicator_flag` remains a historical donor-status field rather than the modeling target. However, its calculation logic and reference date are not currently known, so it will not be used as a predictor unless its timing is confirmed.

The following three columns were classified as excluded features:

- `is_member_flag`
- `current_fiscal_year_donation`
- `cumulative_donation_amount`

`is_member_flag` is excluded because it is constant throughout the finalized dataset and provides no predictive variation.

`current_fiscal_year_donation` directly determines the binary target and would cause target leakage if included as a predictor. It will be retained temporarily only for target construction and validation.

`cumulative_donation_amount` may contain current-fiscal-year donations and therefore may include information from the outcome period. The original column will remain excluded. A separate historical-value feature will be created using only donation information available before the prediction cutoff.

`donor_unique_id` was classified as an identifier. It will be retained for tracking records and connecting predictions back to individual donors, but it will not be included in the predictor matrix.

The final modeling target is `target_current_fiscal_year_donor_flag`.

The leakage audit provides the initial rules for predictor eligibility. Columns classified as potential leakage may be reconsidered if reliable documentation confirms that they were available before the prediction baseline.


In [15]:
# Define feature availability timeline
ESTIMATED_DEMOGRAPHIC_REFERENCE_PERIOD = (
    "Approximately 2018, based on the consistent eight-year difference "
    "between donor_age and age derived from donor_date_of_birth"
)

PREDICTION_BASELINE = (
    "Beginning of the dataset's current fiscal year "
    "(exact fiscal year and calendar date unconfirmed)"
)

HISTORICAL_OBSERVATION_PERIOD = (
    "Five completed fiscal years preceding the prediction baseline"
)

TARGET_OUTCOME_PERIOD = (
    "The dataset's current fiscal year"
)

print("Estimated demographic reference period:")
print(f"  {ESTIMATED_DEMOGRAPHIC_REFERENCE_PERIOD}")

print("\nPrediction baseline:")
print(f"  {PREDICTION_BASELINE}")

print("\nHistorical observation period:")
print(f"  {HISTORICAL_OBSERVATION_PERIOD}")

print("\nTarget outcome period:")
print(f"  {TARGET_OUTCOME_PERIOD}")

Estimated demographic reference period:
  Approximately 2018, based on the consistent eight-year difference between donor_age and age derived from donor_date_of_birth

Prediction baseline:
  Beginning of the dataset's current fiscal year (exact fiscal year and calendar date unconfirmed)

Historical observation period:
  Five completed fiscal years preceding the prediction baseline

Target outcome period:
  The dataset's current fiscal year


In [16]:
# Document feature availability relative to prediction cutoff
feature_availability = pd.DataFrame(
    [
        {
            "column_or_group": "Historical five-year donation columns",
            "availability_relative_to_cutoff": "Before cutoff",
            "feature_status": "Available",
            "notes": (
                "Represent five completed fiscal years before the "
                "target outcome period."
            ),
        },
        {
            "column_or_group": "current_fiscal_year_donation",
            "availability_relative_to_cutoff": "During outcome period",
            "feature_status": "Unavailable as predictor",
            "notes": (
                "Used only to construct and validate the binary target."
            ),
        },
        {
            "column_or_group": "cumulative_donation_amount",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Excluded",
            "notes": (
                "May include current-fiscal-year donations and must not "
                "be used directly."
            ),
        },
        {
            "column_or_group": "consecutive_donor_years",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Requires verification",
            "notes": (
                "May include donation activity from the current fiscal year."
            ),
        },
        {
            "column_or_group": "donor_indicator_flag",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Requires verification",
            "notes": (
                "Calculation logic and reference date are not confirmed."
            ),
        },
        {
            "column_or_group": "Demographic variables",
            "availability_relative_to_cutoff": "Provisionally before cutoff",
            "feature_status": "Provisionally available",
            "notes": (
                "donor_age appears to reflect an approximate 2018 reference "
                "period, but exact field snapshot dates are unconfirmed."
            ),
        },
        {
            "column_or_group": "Stable relationship variables",
            "availability_relative_to_cutoff": "Provisionally before cutoff",
            "feature_status": "Provisionally available",
            "notes": (
                "Includes is_alumnus_flag, is_parent_flag, and "
                "preferred_address_type."
            ),
        },
        {
            "column_or_group": "Engagement and contact variables",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Requires verification",
            "notes": (
                "Snapshot dates for has_involvement_flag and "
                "has_email_flag are not confirmed."
            ),
        },
        {
            "column_or_group": "donor_unique_id",
            "availability_relative_to_cutoff": "Not applicable",
            "feature_status": "Tracking only",
            "notes": (
                "Retained for record identification but excluded from predictors."
            ),
        },
        {
            "column_or_group": "target_current_fiscal_year_donor_flag",
            "availability_relative_to_cutoff": "During outcome period",
            "feature_status": "Target",
            "notes": (
                "Binary modeling outcome derived from "
                "current_fiscal_year_donation."
            ),
        },
    ]
)

display(feature_availability.style.hide(axis="index"))

column_or_group,availability_relative_to_cutoff,feature_status,notes
Historical five-year donation columns,Before cutoff,Available,Represent five completed fiscal years before the target outcome period.
current_fiscal_year_donation,During outcome period,Unavailable as predictor,Used only to construct and validate the binary target.
cumulative_donation_amount,Unknown,Excluded,May include current-fiscal-year donations and must not be used directly.
consecutive_donor_years,Unknown,Requires verification,May include donation activity from the current fiscal year.
donor_indicator_flag,Unknown,Requires verification,Calculation logic and reference date are not confirmed.
Demographic variables,Provisionally before cutoff,Provisionally available,"donor_age appears to reflect an approximate 2018 reference period, but exact field snapshot dates are unconfirmed."
Stable relationship variables,Provisionally before cutoff,Provisionally available,"Includes is_alumnus_flag, is_parent_flag, and preferred_address_type."
Engagement and contact variables,Unknown,Requires verification,Snapshot dates for has_involvement_flag and has_email_flag are not confirmed.
donor_unique_id,Not applicable,Tracking only,Retained for record identification but excluded from predictors.
target_current_fiscal_year_donor_flag,During outcome period,Target,Binary modeling outcome derived from current_fiscal_year_donation.


## Feature Availability Cutoff

To prevent temporal data leakage, every engineered predictor must be based only on information that would have been available when the prediction was made.

The feature engineering process uses the following timeline:

* Estimated demographic reference period: Approximately 2018, based on the consistent difference of eight years between `donor_age` and age derived from `donor_date_of_birth`
* Prediction baseline: The beginning of the dataset’s current fiscal year
* Historical observation period: The five completed fiscal years preceding the prediction baseline
* Target outcome period: The dataset’s current fiscal year

The exact fiscal year and calendar dates represented by the donation fields are not available. Therefore, the prediction cutoff is defined relative to the fiscal year labels in the dataset rather than an assumed calendar date.

During Phase 1, `donor_age` was consistently eight years lower than the age calculated from `donor_date_of_birth` across all valid records. Assuming the derived ages used a 2026 reference date, this suggests that `donor_age` reflects an approximate 2018 demographic reference period.

This finding indicates that at least some fields represent a historical snapshot. However, it does not confirm that the dataset’s current fiscal year is 2018 because demographic, engagement, contact, and donation fields may have been updated on different schedules.

The five historical donation columns are considered available before the prediction cutoff because they represent completed fiscal years preceding the target outcome period.

Based on the feature availability assessment:

* `current_fiscal_year_donation` belongs to the outcome period. It is used only to construct and validate `target_current_fiscal_year_donor_flag` and must not be used as a predictor.
* `cumulative_donation_amount` remains excluded because it may include donations from the current fiscal year. A separate historical value feature will be created using only donation information available before the prediction cutoff.
* `consecutive_donor_years` and `donor_indicator_flag` require additional verification because their calculation periods are unknown and may include activity from the current fiscal year.
* Demographic, geographic, contact preference, and stable relationship fields are provisionally considered available before the cutoff. `has_involvement_flag` and `has_email_flag` will remain unavailable until their snapshot timing is confirmed.
* `donor_unique_id` is retained only for record tracking, while `is_member_flag` remains excluded because it is constant in the finalized dataset.

All subsequent engineered features will be calculated using only information considered available before the prediction baseline.


In [17]:
# Define completed historical donation periods
HISTORICAL_DONATION_COLUMNS = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
]

FEATURE_HISTORICAL_VALUE_COLUMN = (
    "feature_past_5yr_total_donation"
)

# Confirm that all required source columns are available
missing_historical_columns = sorted(
    set(HISTORICAL_DONATION_COLUMNS) - set(df.columns)
)

assert not missing_historical_columns, (
    "Missing historical donation columns: "
    f"{missing_historical_columns}"
)

# Validate the historical donation values
historical_missing_values = (
    df[HISTORICAL_DONATION_COLUMNS]
    .isna()
    .sum()
    .sum()
)

historical_negative_values = (
    df[HISTORICAL_DONATION_COLUMNS] < 0
).sum().sum()

print(
    "Missing historical donation values: "
    f"{historical_missing_values:,}"
)

print(
    "Negative historical donation values: "
    f"{historical_negative_values:,}"
)

assert historical_missing_values == 0, (
    "Historical donation columns contain missing values."
)

assert historical_negative_values == 0, (
    "Historical donation columns contain negative values."
)

Missing historical donation values: 0
Negative historical donation values: 0


In [18]:
# Create historical donation value feature
df[FEATURE_HISTORICAL_VALUE_COLUMN] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .sum(axis=1)
)

print(
    "Historical value feature created: "
    f"{FEATURE_HISTORICAL_VALUE_COLUMN}"
)

Historical value feature created: feature_past_5yr_total_donation


In [19]:
# Validate historical value feature
expected_historical_5yr_total = (
    df[HISTORICAL_DONATION_COLUMNS]
    .sum(axis=1)
)

historical_value_mismatch_count = (
    ~np.isclose(
        df[FEATURE_HISTORICAL_VALUE_COLUMN],
        expected_historical_5yr_total,
    )
).sum()

assert historical_value_mismatch_count == 0, (
    "Found "
    f"{historical_value_mismatch_count:,} "
    "historical value calculation mismatches."
)

assert (
    df[FEATURE_HISTORICAL_VALUE_COLUMN] >= 0
).all(), (
    f"{FEATURE_HISTORICAL_VALUE_COLUMN} "
    "contains negative values."
)

print("Historical value feature validation passed.")

Historical value feature validation passed.


In [20]:
# Summarize historical value feature
historical_value_summary = pd.DataFrame(
    {
        "metric": [
            "Included donation periods",
            "Records",
            "Records with historical donations",
            "Records without historical donations",
            "Mean historical value",
            "Median historical value",
            "Maximum historical value",
        ],
        "value": [
            len(HISTORICAL_DONATION_COLUMNS),
            len(df),
            (
                df[FEATURE_HISTORICAL_VALUE_COLUMN] > 0
            ).sum(),
            (
                df[FEATURE_HISTORICAL_VALUE_COLUMN] == 0
            ).sum(),
            df[FEATURE_HISTORICAL_VALUE_COLUMN].mean(),
            df[FEATURE_HISTORICAL_VALUE_COLUMN].median(),
            df[FEATURE_HISTORICAL_VALUE_COLUMN].max(),
        ],
    }
)

display(historical_value_summary.style.format(
        {
            "value": lambda x: f"{int(x)}" if x % 1 == 0 else f"{x:.2f}"
        }
    ).hide(axis="index"))

metric,value
Included donation periods,5
Records,34403
Records with historical donations,9087
Records without historical donations,25316
Mean historical value,723.44
Median historical value,0
Maximum historical value,10200274


## Historical Donation Value

Based on the feature eligibility assessment, `cumulative_donation_amount` was classified as an excluded feature.

The original `cumulative_donation_amount` column may include donations from the current fiscal year. However, the dataset does not provide transaction dates or a verified fiscal year breakdown for the full cumulative amount. Subtracting `current_fiscal_year_donation` would require assuming that both columns were calculated using the same timing and accounting rules. Because that assumption has not been confirmed, `cumulative_donation_amount` will remain excluded from the predictor set.

Instead, the safer feature `feature_past_5yr_total_donation` was created by summing the five completed historical donation periods:

- `last_fiscal_year_donation`
- `donation_2_fiscal_years_ago`
- `donation_3_fiscal_years_ago`
- `donation_4_fiscal_years_ago`
- `donation_5_fiscal_years_ago`

The current fiscal year donation amount was not included because it belongs to the target outcome period.

`feature_past_5yr_total_donation` represents total giving during the available five year historical observation window. It does not represent the donor’s complete lifetime giving and should not be interpreted as lifetime value.


In [21]:
# Define the basic historical donation feature names
BASIC_HISTORICAL_FEATURE_COLUMNS = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_median_donation",
    "feature_past_5yr_max_donation",
    "feature_past_5yr_min_donation",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_active_average_donation",
]

assert "feature_past_5yr_total_donation" in df.columns, (
    "feature_past_5yr_total_donation was not found. "
    "Restart Kernal and Run all Cells."
)

print("Historical donation source columns:")
print(HISTORICAL_DONATION_COLUMNS)

print("\nBasic historical feature columns:")
print(BASIC_HISTORICAL_FEATURE_COLUMNS)

Historical donation source columns:
['last_fiscal_year_donation', 'donation_2_fiscal_years_ago', 'donation_3_fiscal_years_ago', 'donation_4_fiscal_years_ago', 'donation_5_fiscal_years_ago']

Basic historical feature columns:
['feature_past_5yr_total_donation', 'feature_past_5yr_average_donation', 'feature_past_5yr_median_donation', 'feature_past_5yr_max_donation', 'feature_past_5yr_min_donation', 'feature_past_5yr_donation_std', 'feature_past_5yr_active_average_donation']


In [22]:
# Summary features across the five historical donation periods
df["feature_past_5yr_average_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .mean(axis=1)
)

df["feature_past_5yr_median_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .median(axis=1)
)

df["feature_past_5yr_max_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .max(axis=1)
)

df["feature_past_5yr_min_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .min(axis=1)
)

df["feature_past_5yr_donation_std"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .std(axis=1, ddof=0)
)

positive_historical_donations = (
    df[HISTORICAL_DONATION_COLUMNS]
    .where(df[HISTORICAL_DONATION_COLUMNS] > 0)
)

df["feature_past_5yr_active_average_donation"] = (
    positive_historical_donations
    .mean(axis=1)
    .fillna(0.0)
)

print("Basic historical donation features created.")

Basic historical donation features created.


In [23]:
# Check features for missing, infinite, or negative values
historical_feature_missing_values = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

historical_feature_infinite_values = np.isinf(
    df[BASIC_HISTORICAL_FEATURE_COLUMNS]
).sum().sum()

historical_feature_negative_values = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS] < 0
).sum().sum()

print(
    "Missing engineered historical values: "
    f"{historical_feature_missing_values:,}"
)

print(
    "Infinite engineered historical values: "
    f"{historical_feature_infinite_values:,}"
)

print(
    "Negative engineered historical values: "
    f"{historical_feature_negative_values:,}"
)

assert historical_feature_missing_values == 0, (
    "Historical donation features contain missing values."
)

assert historical_feature_infinite_values == 0, (
    "Historical donation features contain infinite values."
)

assert historical_feature_negative_values == 0, (
    "Historical donation features contain negative values."
)

# Confirm that exactly five historical periods are being used
historical_period_count = len(HISTORICAL_DONATION_COLUMNS)

assert historical_period_count == 5, (
    "Past five year features require exactly five "
    "historical donation columns."
)

# Confirm that the average = the total ÷ the number of historical periods
average_mismatch_count = (
    ~np.isclose(
        df["feature_past_5yr_average_donation"],
        (
            df["feature_past_5yr_total_donation"]
            / historical_period_count
        ),
    )
).sum()

assert average_mismatch_count == 0, (
    f"Found {average_mismatch_count:,} "
    "average calculation mismatches."
)

inactive_average_mismatch_count = (
    (df["feature_past_5yr_total_donation"] == 0)
    & (
        ~np.isclose(
            df["feature_past_5yr_active_average_donation"],
            0.0,
        )
    )
).sum()

assert inactive_average_mismatch_count == 0, (
    "Records without historical donations have unexpected "
    "active average values."
)

# Confirm that excluding inactive years does not lower the average
active_average_order_mismatch_count = (
    df["feature_past_5yr_active_average_donation"]
    + 1e-10
    < df["feature_past_5yr_average_donation"]
).sum()

assert active_average_order_mismatch_count == 0, (
    "Active donation averages were unexpectedly lower than "
    "overall historical averages."
)

print("Basic historical donation feature validation passed.")

Missing engineered historical values: 0
Infinite engineered historical values: 0
Negative engineered historical values: 0
Basic historical donation feature validation passed.


In [24]:
# Summarize the basic historical donation features
basic_historical_feature_summary = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS]
    .describe()
    .transpose()
)

basic_historical_feature_summary["nonzero_records"] = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS] > 0
).sum()

basic_historical_feature_summary.index.name = "feature"

display(
    basic_historical_feature_summary.style.format(
        lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    )
)

,count,mean,std,min,25%,50%,75%,max,nonzero_records
feature,,,,,,,,,
feature_past_5yr_total_donation,"34,403",723.44,"58,342.93",0,0,0,1,"10,200,274","9,087"
feature_past_5yr_average_donation,"34,403",144.69,"11,668.59",0,0,0,0.2,"2,040,054.8","9,087"
feature_past_5yr_median_donation,"34,403",17.27,727.13,0,0,0,0,"90,000",407
feature_past_5yr_max_donation,"34,403",596.12,"55,992.71",0,0,0,1,"10,000,000","9,087"
feature_past_5yr_min_donation,"34,403",1.6,148.9,0,0,0,0,"25,000",10
feature_past_5yr_donation_std,"34,403",236.14,"22,261.85",0,0,0,0.4,"3,980,724.03","9,087"
feature_past_5yr_active_average_donation,"34,403",241.29,"15,401.43",0,0,0,1,"2,550,068.5","9,087"


## Historical Donation Summary Features

Basic donation value features were created using only the five completed fiscal years before the prediction period.

The previously created `feature_past_5yr_total_donation` feature was retained and used alongside six additional features:

- `feature_past_5yr_average_donation`
- `feature_past_5yr_median_donation`
- `feature_past_5yr_max_donation`
- `feature_past_5yr_min_donation`
- `feature_past_5yr_donation_std`
- `feature_past_5yr_active_average_donation`

The engineered features are:

- The overall average represents the average donation amount across all five historical years, including years with no donation.
- The median, maximum, and minimum summarize annual donation amounts within the historical observation window. Because zero represents a year without a donation, the median and minimum are zero for most individuals. The minimum is greater than zero only when an individual donated during all five historical years.
- The standard deviation measures how much annual donation amounts varied across the five year period. A population standard deviation was used because the calculation covers the complete historical observation window available in the dataset.
- The active average was calculated using only years in which the individual donated more than zero. It represents the typical amount donated during active giving years rather than averaging across inactive years. Individuals with no historical donations were assigned an active average of zero.

The feature distributions remain highly right skewed because a small number of donors contributed exceptionally large amounts. Skew resistant transformations will be created in a later feature engineering step.

No current fiscal year donation values or cumulative donation values were used to create these features.


In [25]:
# Define historical donation frequency feature
DONATION_FREQUENCY_FEATURE_COLUMNS = [
    "feature_years_donated_past_5yr",
    "feature_past_5yr_donation_frequency_rate",
    "feature_any_past_donation_flag",
    "feature_multiple_year_donor_flag",
    "feature_consistent_donor_flag",
]

historical_period_count = len(HISTORICAL_DONATION_COLUMNS)

assert historical_period_count == 5, (
    "Past five year frequency features require exactly five "
    "historical donation columns."
)

print("Donation frequency feature columns:")
print(DONATION_FREQUENCY_FEATURE_COLUMNS)

Donation frequency feature columns:
['feature_years_donated_past_5yr', 'feature_past_5yr_donation_frequency_rate', 'feature_any_past_donation_flag', 'feature_multiple_year_donor_flag', 'feature_consistent_donor_flag']


In [26]:
# Donation freq. features
historical_donation_activity = (
    df[HISTORICAL_DONATION_COLUMNS] > 0
)

df["feature_years_donated_past_5yr"] = (
    historical_donation_activity
    .sum(axis=1)
    .astype("int8")
)

df["feature_past_5yr_donation_frequency_rate"] = (
    df["feature_years_donated_past_5yr"]
    / historical_period_count
)

df["feature_any_past_donation_flag"] = (
    df["feature_years_donated_past_5yr"] >= 1
).astype("int8")

df["feature_multiple_year_donor_flag"] = (
    df["feature_years_donated_past_5yr"] >= 2
).astype("int8")

df["feature_consistent_donor_flag"] = (
    df["feature_years_donated_past_5yr"] >= 3
).astype("int8")

print("Donation frequency features created.")

Donation frequency features created.


In [27]:
# Validate donation freq. features
frequency_feature_missing_values = (
    df[DONATION_FREQUENCY_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

frequency_feature_infinite_values = np.isinf(
    df[DONATION_FREQUENCY_FEATURE_COLUMNS]
).sum().sum()

frequency_feature_negative_values = (
    df[DONATION_FREQUENCY_FEATURE_COLUMNS] < 0
).sum().sum()

print(
    "Missing donation frequency values: "
    f"{frequency_feature_missing_values:,}"
)

print(
    "Infinite donation frequency values: "
    f"{frequency_feature_infinite_values:,}"
)

print(
    "Negative donation frequency values: "
    f"{frequency_feature_negative_values:,}"
)

assert frequency_feature_missing_values == 0, (
    "Donation frequency features contain missing values."
)

assert frequency_feature_infinite_values == 0, (
    "Donation frequency features contain infinite values."
)

assert frequency_feature_negative_values == 0, (
    "Donation frequency features contain negative values."
)

# Confirm num of active years is in between 0 and 5
assert df["feature_years_donated_past_5yr"].between(
    0,
    historical_period_count,
).all(), (
    "Historical donation year counts fall outside the expected range."
)

# Confirm frequency rate falls between 0 and 1
assert df["feature_past_5yr_donation_frequency_rate"].between(
    0,
    1,
).all(), (
    "Donation frequency rates fall outside the expected range."
)

# Confirm frequency rate matches the active year count
frequency_rate_mismatch_count = (
    ~np.isclose(
        df["feature_past_5yr_donation_frequency_rate"],
        (
            df["feature_years_donated_past_5yr"]
            / historical_period_count
        ),
    )
).sum()

assert frequency_rate_mismatch_count == 0, (
    f"Found {frequency_rate_mismatch_count:,} "
    "donation frequency rate mismatches."
)

any_donation_mismatch_count = (
    df["feature_any_past_donation_flag"]
    != (
        df["feature_years_donated_past_5yr"] >= 1
    ).astype("int8")
).sum()

multiple_year_mismatch_count = (
    df["feature_multiple_year_donor_flag"]
    != (
        df["feature_years_donated_past_5yr"] >= 2
    ).astype("int8")
).sum()

consistent_donor_mismatch_count = (
    df["feature_consistent_donor_flag"]
    != (
        df["feature_years_donated_past_5yr"] >= 3
    ).astype("int8")
).sum()

assert any_donation_mismatch_count == 0, (
    f"Found {any_donation_mismatch_count:,} "
    "any donation flag mismatches."
)

assert multiple_year_mismatch_count == 0, (
    f"Found {multiple_year_mismatch_count:,} "
    "multiple year donor flag mismatches."
)

assert consistent_donor_mismatch_count == 0, (
    f"Found {consistent_donor_mismatch_count:,} "
    "consistent donor flag mismatches."
)

frequency_hierarchy_mismatch_count = (
    (
        df["feature_consistent_donor_flag"]
        > df["feature_multiple_year_donor_flag"]
    )
    | (
        df["feature_multiple_year_donor_flag"]
        > df["feature_any_past_donation_flag"]
    )
).sum()

assert frequency_hierarchy_mismatch_count == 0, (
    "Donation frequency flags do not follow the expected hierarchy."
)

print("Donation frequency feature validation passed.")

Missing donation frequency values: 0
Infinite donation frequency values: 0
Negative donation frequency values: 0
Donation frequency feature validation passed.


In [28]:
# Summarize distribution of active donation years
donation_frequency_distribution = (
    df["feature_years_donated_past_5yr"]
    .value_counts()
    .sort_index()
    .rename_axis("years_donated_past_5yr")
    .reset_index(name="record_count")
)

donation_frequency_distribution["percentage"] = (
    donation_frequency_distribution["record_count"]
    / len(df)
    * 100
).round(2)

display(
    donation_frequency_distribution.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "record_count": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

years_donated_past_5yr,record_count,percentage
0,"25,316",73.59%
1,"7,193",20.91%
2,"1,487",4.32%
3,347,1.01%
4,50,0.15%
5,10,0.03%


In [29]:
# Summarize the binary donation frequency indicators
donation_frequency_flag_summary = pd.DataFrame(
    {
        "feature": [
            "feature_any_past_donation_flag",
            "feature_multiple_year_donor_flag",
            "feature_consistent_donor_flag",
        ],
        "records_flagged": [
            df["feature_any_past_donation_flag"].sum(),
            df["feature_multiple_year_donor_flag"].sum(),
            df["feature_consistent_donor_flag"].sum(),
        ],
    }
)

donation_frequency_flag_summary["percentage"] = (
    donation_frequency_flag_summary["records_flagged"]
    / len(df)
    * 100
).round(2)

display(
    donation_frequency_flag_summary.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "records_flagged": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

feature,records_flagged,percentage
feature_any_past_donation_flag,"9,087",26.41%
feature_multiple_year_donor_flag,"1,894",5.51%
feature_consistent_donor_flag,407,1.18%


## Historical Donation Frequency

Donation frequency features were created using the five completed fiscal years before the target outcome period.

The following features were created:
- `feature_years_donated_past_5yr` counts the number of historical years in which an individual donated more than zero.
- `feature_past_5yr_donation_frequency_rate` divides the number of active donation years by five. The feature ranges from 0 to 1, where 0 represents no historical donations and 1 represents donations during all five historical years.
- `feature_any_past_donation_flag` identifies individuals who donated during at least one historical year.
- `feature_multiple_year_donor_flag` identifies individuals who donated during at least two historical years.
- `feature_consistent_donor_flag` identifies individuals who donated during at least three historical years.

The binary thresholds provide simple donor frequency groups, but they are not treated as final assumptions about donor behavior. These features measure how often an individual donated, but they do not describe whether the donations occurred consecutively.

No current fiscal year donation information was used to create these features.
